In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

sentences = [
    "I love machine learning.",
    "Artificial intelligence is fascinating.",
    "The weather is beautiful today."
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_embeddings(model_name, sentences, causal=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    # GPT-2 requires a padding token for batched inputs
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.pad_token_id

    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        outputs = model(**inputs)
        hidden = outputs.last_hidden_state

        # Mean pooling; exclude padding tokens
        mask = inputs["attention_mask"].unsqueeze(-1)
        embeddings = (hidden * mask).sum(dim=1)
        embeddings = embeddings / mask.sum(dim=1).clamp(min=1)

        # Unit-normalized embeddings
        embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings


# BERT embeddings
bert_embeddings = get_embeddings(
    "google-bert/bert-base-uncased",
    sentences
)

# GPT-2 embeddings
gpt2_embeddings = get_embeddings(
    "openai-community/gpt2",
    sentences
)

# Cosine similarity matrices
bert_similarity = bert_embeddings @ bert_embeddings.T
gpt2_similarity = gpt2_embeddings @ gpt2_embeddings.T

print("BERT Embeddings:", bert_embeddings.shape)
print("GPT-2 Embeddings:", gpt2_embeddings.shape)

print("\nBERT Similarity:\n", bert_similarity.cpu().numpy().round(3))
print("\nGPT-2 Similarity:\n", gpt2_similarity.cpu().numpy().round(3))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BERT Embeddings: torch.Size([3, 768])
GPT-2 Embeddings: torch.Size([3, 768])

BERT Similarity:
 [[1.    0.788 0.625]
 [0.788 1.    0.666]
 [0.625 0.666 1.   ]]

GPT-2 Similarity:
 [[1.    0.997 0.998]
 [0.997 1.    0.998]
 [0.998 0.998 1.   ]]
